<div style="text-align:center; padding-top:60px;">

# Practical Deep Learning
## Data Augmentation · Generators · Regularization

**Image Classification in Practice**

**MUSA 650 — Week 10**

---
⏱ 75 minutes &nbsp;|&nbsp; 4 parts &nbsp;|&nbsp; live code

</div>

## Recap: Weeks 8 & 9

| Week | Topics |
|------|--------|
| 8 | Neural networks basics, MLP, first CNNs on MNIST |
| 9 | CNN architecture deep-dive, image segmentation, U-Net |
| **10** | **Making DL work in practice: augmentation, generators, regularization** |

---

**The gap between a working notebook and a working model:**
- Week 8–9: We built models that *run*
- Week 10: We make models that *generalize*

## Today's Agenda

| # | Topic | Time |
|---|-------|------|
| 1 | The Overfitting Problem | ~10 min |
| 2 | Data Augmentation | ~20 min |
| 3 | Data Generators | ~10 min |
| 4 | Dropout & Batch Normalization | ~20 min |
| 5 | Putting It All Together + Lab | ~15 min |

> **Goal**: understand why models fail on new data and how to fix it with practical Keras tools.

# Part 1
## The Overfitting Problem

## What is Overfitting?

A model **overfits** when it learns the training data *too well* — including noise — and fails to generalize to new data.

```
Training accuracy:   95%   ← model memorized training set
Validation accuracy: 65%   ← model fails on new images
```

### Why does it happen with images?

| Cause | Example |
|-------|---------|
| **Too few training images** | 200 cat photos → model memorizes them |
| **Model too complex** | 10M parameters, 500 training samples |
| **No regularization** | Nothing prevents weights from growing large |
| **No diversity** | All cat photos taken from same angle |

> Rule of thumb: for image classification, you typically need **thousands** of images per class.

## Detecting Overfitting: Training Curves

The clearest signal is the **gap between training and validation loss**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epochs = np.arange(1, 21)
train_loss = 0.6 * np.exp(-0.15 * epochs) + 0.05
val_loss   = 0.6 * np.exp(-0.08 * epochs) + 0.25 + 0.01 * epochs

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(epochs, train_loss, 'b-o', ms=4, label='Training loss')
ax.plot(epochs, val_loss,   'r-o', ms=4, label='Validation loss')
ax.axvspan(12, 20, alpha=0.1, color='red')
ax.text(13, 0.55, 'Overfitting\nzone', color='red', fontsize=10)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Overfitting: loss diverges')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

train_acc = 1 - train_loss
val_acc   = 1 - val_loss
ax2 = axes[1]
ax2.plot(epochs, train_acc, 'b-o', ms=4, label='Training acc')
ax2.plot(epochs, val_acc,   'r-o', ms=4, label='Validation acc')
ax2.axvspan(12, 20, alpha=0.1, color='red')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Overfitting: accuracy gap widens')
ax2.legend(); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Our Toolkit Against Overfitting

| Technique | What it does | When to use |
|-----------|-------------|-------------|
| **Data Augmentation** | Artificially increases training set diversity | Always — especially with small datasets |
| **Data Generators** | Efficiently feeds data to the model | When dataset doesn't fit in RAM |
| **Dropout** | Randomly disables neurons during training | After dense/conv layers |
| **Batch Normalization** | Normalizes layer inputs during training | After conv layers |
| Early stopping | Stops training when validation stops improving | When you have patience callbacks |
| L1/L2 regularization | Penalizes large weights | When other methods aren't enough |

**Today we focus on the first four.**

# Part 2
## Data Augmentation

## Data Augmentation: The Core Idea

**Problem**: We don't have enough training images.

**Solution**: Generate *new* training images by applying random, realistic transformations to existing ones.

```
Original cat photo
       │
   ┌───┼────────────────────┐
   ↓   ↓         ↓          ↓
Flip  Rotate  Zoom in   Shift left
 (still a cat)  (still a cat)  ...
```

**Key insight**: The *label* doesn't change — a horizontally flipped cat is still a cat.

> Augmentation is applied **on-the-fly during training**, so disk storage doesn't grow.

## Common Augmentation Types

| Augmentation | Keras parameter | Typical value |
|---|---|---|
| Horizontal flip | `horizontal_flip=True` | Boolean |
| Rotation | `rotation_range=20` | Degrees (0–180) |
| Width/height shift | `width_shift_range=0.2` | Fraction of width |
| Zoom | `zoom_range=0.2` | Fraction |
| Brightness | `brightness_range=[0.8, 1.2]` | Min/max multiplier |
| Shear | `shear_range=0.1` | Shear intensity |
| Fill mode | `fill_mode='nearest'` | How to fill empty pixels |

**What NOT to augment** (label-breaking transforms):
- 180° rotation of a digit **6** → looks like **9**
- Vertical flip of satellite imagery (sometimes)
- Color inversion of medical images

## Augmentation with Keras: `ImageDataGenerator`

```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)
```

Then use it during training:

```python
# Option A: fit() with generator
model.fit(datagen.flow(x_train, y_train, batch_size=32),
          epochs=20,
          validation_data=(x_val, y_val))

# Option B: flow_from_directory (reads images from disk)
train_gen = datagen.flow_from_directory('data/train/', target_size=(64,64))
```

## Live Demo: Augmenting Cat Images

We'll load a few cat images from CIFAR-10 and visualize augmented versions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load CIFAR-10; class 3 = cats
(x_train, y_train), _ = cifar10.load_data()
cat_images = x_train[y_train.flatten() == 3][:5]   # 5 cat images
print("Cat image shape:", cat_images[0].shape)      # (32, 32, 3)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.3,
    fill_mode='nearest'
)

# Show original + 5 augmented versions of the first cat
original = cat_images[0]
fig, axes = plt.subplots(2, 6, figsize=(13, 5))

for row, cat in enumerate(cat_images[:2]):
    axes[row, 0].imshow(cat)
    axes[row, 0].set_title('Original', fontsize=9)
    axes[row, 0].axis('off')
    img_batch = cat.reshape((1,) + cat.shape)
    for col, batch in zip(range(1, 6), datagen.flow(img_batch, batch_size=1)):
        axes[row, col].imshow(batch[0].astype('uint8'))
        axes[row, col].set_title(f'Aug {col}', fontsize=9)
        axes[row, col].axis('off')

plt.suptitle('CIFAR-10 Cats — Original vs Augmented', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Part 3
## Data Generators

## Why Do We Need Data Generators?

**Problem**: Real datasets are large — they don't fit in RAM.

```
1,000 images × 224×224×3 pixels × 4 bytes = ~600 MB
1,000,000 images                           = ~600 GB  ← won't fit in RAM!
```

**Solution**: Load data **in batches** as the model needs it.

### Generator vs. loading everything up front

| Approach | Memory | Speed | Augmentation |
|----------|--------|-------|--------------|
| `np.load()` all at once | High | Fast (already loaded) | Manual |
| **Generator** | Low (one batch at a time) | Slightly slower (I/O) | Built-in |

> Generators are also *iterable* — Keras knows how to call them during `model.fit()`.

## `flow_from_directory`: Reading Images from Disk

This is the most common pattern for real projects:

```
data/
├── train/
│   ├── cats/
│   │   ├── cat001.jpg
│   │   └── cat002.jpg
│   └── dogs/
│       ├── dog001.jpg
│       └── dog002.jpg
└── validation/
    ├── cats/
    └── dogs/
```

```python
datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True)

train_gen = datagen.flow_from_directory(
    'data/train/',
    target_size=(64, 64),   # resize all images to this size
    batch_size=32,
    class_mode='binary'     # or 'categorical' for multi-class
)

model.fit(train_gen, epochs=20, validation_data=val_gen)
```

> **Folder names become class labels automatically.**

## `flow()`: Generator from Numpy Arrays

When data is already in memory (like MNIST/CIFAR-10):

```python
datagen = ImageDataGenerator(
    rotation_range=15,
    horizontal_flip=True,
    rescale=1./255
)

# Fit the generator on your data (needed for featurewise normalization)
datagen.fit(x_train)

# Create the generator
train_gen = datagen.flow(x_train, y_train, batch_size=32)

# Train with it
model.fit(train_gen,
          steps_per_epoch=len(x_train) // 32,   # batches per epoch
          epochs=20,
          validation_data=(x_val, y_val))
```

**`steps_per_epoch`**: how many batches = one epoch  
Rule: `steps_per_epoch = num_samples // batch_size`

# Part 4
## Dropout & Batch Normalization

## Dropout: Randomly Switching Off Neurons

**Idea**: During each training step, randomly set a fraction of neuron outputs to **zero**.

```
Without dropout:       With dropout (rate=0.5):
  ●─●─●─●             ●─●─○─●   ← 2 neurons disabled
  │ │ │ │             │ │   │
  ●─●─●─●             ○─●─●─○   ← 2 neurons disabled
```

### Why does this help?
- Forces the network to learn **redundant representations**
- Prevents neurons from co-adapting too much
- Acts like training an **ensemble of smaller networks**
- During **inference** (prediction), all neurons are active but outputs are scaled

> Dropout is applied **only during training**, not during prediction.

## Dropout in Keras

```python
from tensorflow.keras.layers import Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D(2,2),
    Dropout(0.25),           # drop 25% of conv layer outputs

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),            # drop 50% before the output layer
    Dense(1, activation='sigmoid')
])
```

### Typical dropout rates:
| Layer type | Typical rate |
|---|---|
| After Conv layers | 0.1 – 0.3 |
| After Dense (hidden) | 0.3 – 0.5 |
| After final Dense | Rarely used |

## Batch Normalization: Stabilizing Layer Inputs

**Problem**: As training progresses, the distribution of each layer's inputs changes — this slows learning (called *internal covariate shift*).

**Solution**: Normalize the outputs of a layer to have **mean ≈ 0** and **std ≈ 1** for each mini-batch.

```
Conv2D output:   [23, -150, 0.01, 8900, ...]   ← wildly different scales
After BatchNorm: [-0.3, -1.1, 0.2, 1.4, ...]  ← normalized
```

### Benefits:
- **Faster training** (allows higher learning rates)
- **Regularization effect** (slightly reduces overfitting)
- **Less sensitive** to weight initialization
- Makes training more stable — gradients don't explode or vanish as easily

> BatchNorm learns two parameters per feature: **scale** (γ) and **shift** (β).

## Batch Normalization in Keras

```python
from tensorflow.keras.layers import BatchNormalization

model = Sequential([
    Conv2D(32, (3,3), input_shape=(32,32,3)),
    BatchNormalization(),     # normalize BEFORE activation
    Activation('relu'),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3)),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128),
    BatchNormalization(),
    Activation('relu'),
    Dense(1, activation='sigmoid')
])
```

> **Convention debate**: BN before or after activation?  
> Original paper: before. In practice: both work — try both.

> BatchNorm has **different behavior** during training vs inference (uses running mean/std at inference time).

## Dropout vs. Batch Normalization: Quick Comparison

| | Dropout | Batch Normalization |
|---|---|---|
| **Primary goal** | Reduce overfitting | Stabilize/speed up training |
| **How** | Randomly zeroes activations | Normalizes activations |
| **Training vs inference** | Different (active only in training) | Different (uses running stats) |
| **Works with CNNs?** | Yes (usually lower rate) | Yes (very common in CNNs) |
| **Combine?** | Yes — often used together |

### Note on using both:
Some architectures (like ResNet) use BatchNorm but **not** Dropout.  
Using both is fine — start with BN, add Dropout if still overfitting.

## Putting It All Together

A practical recipe for image classification with a small dataset:

```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, Activation
)

# 1. Data augmentation
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20, horizontal_flip=True, zoom_range=0.2
)

# 2. Model with BatchNorm + Dropout
model = Sequential([
    Conv2D(32, (3,3), input_shape=(32,32,3)),
    BatchNormalization(), Activation('relu'),
    MaxPooling2D(2,2), Dropout(0.25),

    Conv2D(64, (3,3)),
    BatchNormalization(), Activation('relu'),
    MaxPooling2D(2,2), Dropout(0.25),

    Flatten(),
    Dense(128), BatchNormalization(), Activation('relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')   # binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(datagen.flow(x_train, y_train, batch_size=32), epochs=30, validation_data=(x_val, y_val))
```

## Summary

| Technique | Keras class/method | Key hyperparameter |
|-----------|-------------------|-------------------|
| Data augmentation | `ImageDataGenerator` | `rotation_range`, `flip`, `zoom_range` |
| Generator from arrays | `.flow(x, y)` | `batch_size` |
| Generator from disk | `.flow_from_directory()` | `target_size`, `class_mode` |
| Dropout | `Dropout(rate)` | `rate` (0.0–1.0) |
| Batch Normalization | `BatchNormalization()` | (mostly parameter-free) |

### Key takeaways
1. **Always** check for overfitting by plotting training vs validation curves
2. **Augmentation** is the easiest win when you have few images
3. **BatchNorm** almost always helps — add it after conv layers
4. **Dropout** after dense layers reduces overfitting; lower rates after conv
5. These techniques **stack** — use them together

---
**Up next → Lab**: apply all of this to cats vs dogs on CIFAR-10